# SearchLite-RL: supervised warm-start + GRPO study (A existing | B SFT | C SFT+GRPO | D SFT+evidence GRPO)
GPU runtime (L4). Nothing here deletes or overwrites earlier results. The pipeline stops by itself if the SFT warm-start fails its gate (exit code 3). Cell 2 restarts the runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!if [ -d Pilot-project ]; then cd Pilot-project && git fetch -q && git checkout -q experiment/evidence-aware-grpo && git pull -q; else git clone -q -b experiment/evidence-aware-grpo https://github.com/aasritha-oql7872/Pilot-project.git; fi
%cd /content/Pilot-project
!pip install -q -r requirements.txt
!pip uninstall -y -q torchvision torchaudio torchao
import os; os.kill(os.getpid(), 9)

In [ ]:
%cd /content/Pilot-project
!nvidia-smi | head -12
# Persist everything to Drive: outputs/study_sft is a symlink into Drive, so adapters, logs and evals survive a disconnect.
# Re-running the notebook resumes (finished stages are skipped; nothing is deleted or overwritten).
!mkdir -p /content/drive/MyDrive/SearchLite-RL/study_sft outputs
!ln -sfn /content/drive/MyDrive/SearchLite-RL/study_sft outputs/study_sft
# Restore condition A (existing one-hop adapter) if you saved it; otherwise the script retrains it (~10 min).
!mkdir -p outputs/main_v2
!cp -rn /content/drive/MyDrive/SearchLite-RL/outputs/main_v2/adapter outputs/main_v2/ 2>/dev/null || cp -rn /content/drive/MyDrive/SearchLite-RL/main_v2/adapter outputs/main_v2/ 2>/dev/null || echo "no saved adapter -> script retrains A (outputs/main_v2_retrained)"
!ls outputs/main_v2 outputs/study_sft


In [ ]:
# Runs tests, SFT data/mask check, A (both reminders), B, dev evals and the GATE, then C and D ONLY if B passes.
# If the gate fails the script stops with "GATE FAILED": paste that output, do not continue.
!set -o pipefail; bash run_sft_study.sh 2>&1 | tee /content/drive/MyDrive/SearchLite-RL/study_sft/run_$(date +%Y%m%d_%H%M).log | tail -150


In [ ]:
# Everything is already on Drive (outputs/study_sft). Also keep the config used.
!cp -n configs/*.yaml /content/drive/MyDrive/SearchLite-RL/study_sft/ ; git rev-parse HEAD > /content/drive/MyDrive/SearchLite-RL/study_sft/git_commit.txt
!cat outputs/study_sft/summary_dev.md || echo "no summary yet (gate failed or run incomplete)"


In [ ]:
# Sealed holdouts: ONLY after you have chosen the method from the dev table. Write the choice, then run once.
# !echo "chosen method: <B|C|D>, chosen on dev table, $(date)" > outputs/study_sft/FROZEN
# !FINAL=1 bash run_sft_study.sh 2>&1 | tail -20
# !cat outputs/study_sft/summary_sealed.md